In [ ]:
! unzip all.zip

In [1]:
! pip install folktables


In [2]:
import torch
import torch.nn.functional as F
from data_generation.BetaVAE import *
from folktables import ACSDataSource, ACSIncome
import pandas as pd
from loaders.vae_loader import *

def load_acs_ca_year(year: int):
    data_source = ACSDataSource(survey_year=str(year), horizon="1-Year", survey="person")
    acs_data = data_source.get_data(states=["CA"], download=True)


    features = ACSIncome.features
    X, y, _ = ACSIncome.df_to_numpy(acs_data)

    X = pd.DataFrame(X, columns=features)
    y = pd.Series(y.astype(int), name="income")
    return X, y

def make_train_test_split_like_paper():
    X14, y14 = load_acs_ca_year(2014)
    X18, y18 = load_acs_ca_year(2018)

    majority_mask_18 = (y18 == 0)

    X_train = pd.concat([X14, X18[majority_mask_18]], axis=0).reset_index(drop=True)
    y_train = pd.concat([y14, y18[majority_mask_18]], axis=0).reset_index(drop=True)

    X_test = X18.reset_index(drop=True)
    y_test = y18.reset_index(drop=True)
    return X_train, y_train, X_test, y_test



In [10]:

# --- CONFIGURATION ---
VAE_CHECKPOINT = "/content/checkpoints/best_vae.pt"
LATENT_DIM = 320  # À ajuster selon ton VAE
EPOCHS = 100
BATCH_SIZE = 512
LR = 1e-3
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [11]:
X_train, y_train, X_test, y_test = make_train_test_split_like_paper()  # CA 2014 + maj 2018 / test = 2018 :contentReference[oaicite:1]{index=1}


cat_cols = [c for c in X_train.columns if X_train[c].dtype == "object"]
num_cols = [c for c in X_train.columns if c not in cat_cols]


df_train = X_train.copy()
df_train["y"] = y_train.values

df_test = X_test.copy()
df_test["y"] = y_test.values


loaders_train, prepro, meta = make_tabular_loaders(
    df=df_train,
    num_cols=num_cols,
    cat_cols=cat_cols,
    label_col="y",
    batch_size=BATCH_SIZE,
    test_size=None,
    val_size=0.1,
)

train_loader = loaders_train["train"]
val_loader   = loaders_train["val"]




In [12]:
num_numerical = len(num_cols)
cat_cardinalities = meta["cat_cardinalities"]

d = 32  # embedding dim (à choisir)
beta = 1.0

In [13]:
train_loader.dataset

In [14]:
X_train_numpy = X_train.to_numpy()
y_train_numpy = y_train.to_numpy()

In [15]:
import torch
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from models.diffusion_model import DenoisingMLP, GaussianDiffusion

# import vae
from data_generation.BetaVAE import BetaVAE


# --- 1. CHARGEMENT DU VAE & EXTRACTION DES LATENTS ---
def get_latent_data(vae, data_loader):
    vae.eval()
    latents = []
    labels = []

    print("Extraction des latents via le VAE...")
    with torch.no_grad():
        for batch in data_loader:
            # 1. Extraction robuste de X et Y
            if isinstance(batch, (list, tuple)):
                x_batch = batch[0]
                y_batch = batch[1] # On suppose que le label est en 2ème position
            else:
                x_batch = batch
                y_batch = None

            x_batch = x_batch.to(DEVICE)

            # 2. Passage 2D -> 3D pour le Transformer
            if x_batch.dim() == 2:
                x_batch = x_batch.unsqueeze(-1)

            # 3. Encodage
            mu, _ = vae.encoder(x_batch)
            latents.append(mu.cpu())

            # 4. Stockage des labels en s'assurant qu'ils sont 1D
            if y_batch is not None:
                labels.append(y_batch.cpu().view(-1)) # .view(-1) force la forme [Batch]

    return torch.cat(latents), torch.cat(labels)




def train_one_model(model_name, diffusion, data_loader, optimizer):
    """Boucle d'entraînement pour un modèle spécifique"""
    print(f"--- Entraînement Modèle {model_name} ---")
    for epoch in range(EPOCHS):
        total_loss = 0
        for z_batch in data_loader:
            z_batch = z_batch[0].to(DEVICE) if isinstance(z_batch, (list, tuple)) else z_batch.to(DEVICE)
            optimizer.zero_grad()
            loss = diffusion.p_loss(z_batch)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        if epoch % 10 == 0:
            print(f"Epoch {epoch}: Loss {total_loss / len(data_loader):.4f}")


# A. Charger le VAE (Adapte selon ton code VAE)
# from models import VAE
vae = BetaVAE(num_numerical=num_numerical, cat_cardinalities=cat_cardinalities, d=d, beta=beta)
checkpoint = torch.load(VAE_CHECKPOINT, map_location=DEVICE)
vae.load_state_dict(checkpoint["vae_state"])
vae.to(DEVICE)

#B.
# X_train = X_train.to_numpy()
# y_train = y_train.to_numpy()
# X_train_tensor = torch.from_numpy(X_train).float()
# y_train_tensor = torch.tensor(y_train)

# Création du DataLoader temporaire pour extraction
full_loader = DataLoader(TensorDataset(torch.from_numpy(X_train_numpy).float(), torch.from_numpy(y_train_numpy)), batch_size=BATCH_SIZE, shuffle=False)
# --- C. Extraction et Correction des formes ---
z_all, y_all = get_latent_data(vae, full_loader)

# Aplatir les latents pour la diffusion : [269343, 10, 32] -> [269343, 320]
z_all = z_all.flatten(1)

# Vérification de sécurité
if z_all.shape[0] != y_all.shape[0]:
    print(f"Attention ! Taille mismatch: z={z_all.shape[0]}, y={y_all.shape[0]}")
    # Au cas où, on tronque pour que ça corresponde
    min_size = min(z_all.shape[0], y_all.shape[0])
    z_all, y_all = z_all[:min_size], y_all[:min_size]

# --- D. Séparation ---
z_normal = z_all[y_all == 0]
z_attack = z_all[y_all == 1]

loader_normal = DataLoader(TensorDataset(z_normal), batch_size=BATCH_SIZE, shuffle=True)
loader_attack = DataLoader(TensorDataset(z_attack), batch_size=BATCH_SIZE, shuffle=True)

# E. Initialisation des modèles de diffusion
model_normal = DenoisingMLP(latent_dim=LATENT_DIM).to(DEVICE)
diff_normal = GaussianDiffusion(model_normal, device=DEVICE)
opt_normal = torch.optim.Adam(model_normal.parameters(), lr=LR)

model_attack = DenoisingMLP(latent_dim=LATENT_DIM).to(DEVICE)
diff_attack = GaussianDiffusion(model_attack, device=DEVICE)
opt_attack = torch.optim.Adam(model_attack.parameters(), lr=LR)

# F. Lancer l'entraînement
train_one_model("NORMAL (S0)", diff_normal, loader_normal, opt_normal)
train_one_model("ATTACK (S1)", diff_attack, loader_attack, opt_attack)

# G. Sauvegarder
torch.save(model_normal.state_dict(), "checkpoints/diffusion_normal.pt")
torch.save(model_attack.state_dict(), "checkpoints/diffusion_attack.pt")
print("Modèles sauvegardés !")



/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Extraction des latents via le VAE...
--- Entraînement Modèle NORMAL (S0) ---
Epoch 0: Loss 0.3353
Epoch 10: Loss 0.2171
Epoch 20: Loss 0.2129
Epoch 30: Loss 0.2122
Epoch 40: Loss 0.2122
Epoch 50: Loss 0.2121
Epoch 60: Loss 0.2110
Epoch 70: Loss 0.2104
Epoch 80: Loss 0.2105
Epoch 90: Loss 0.2105
--- Entraînement Modèle ATTACK (S1) ---
Epoch 0: Loss 0.5241
Epoch 10: Loss 0.2216
Epoch 20: Loss 0.2194
Epoch 30: Loss 0.2179
Epoch 40: Loss 0.2157
Epoch 50: Loss 0.2134
Epoch 60: Loss 0.2134
Epoch 70: Loss 0.2135
Epoch 80: Loss 0.2128
Epoch 90: Loss 0.2122
Modèles sauvegardés !


In [17]:
print(num_numerical,cat_cardinalities, d, beta)

1.0 [] 32 1.0
